# Silver layer

## Product Information

Table to store product information, and it's historic information (older models of the same product):
* Id
* Key
* Full name
* Production cost
* Production line
* Production start
* Production end

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

## Read table from bronze layer

In [0]:
df = spark.read.table("db_project.bronze.crm_prd_info")
df.display()

## Correct Strings

For each string type column, apply trim to remove unnecessary spaces

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        display(field.dataType)
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

## Check dates

In [0]:
test_dates = df.select(
    "prd_id",
    "prd_key", 
    "prd_start_dt", 
    "prd_end_dt"
)
test_dates.display()

A lot of products end of production dates happen to be before the start of new production (newer model), so it will be overwritten with lead() of production start date of newer version of the product minus one day

In [0]:
# df.groupBy("prd_id").count().display()

df = df.withColumn(
    "prd_end_dt", 
    date_add(
        (lead("prd_start_dt", 1).over(
                Window.
                partitionBy("prd_key").
                orderBy("prd_start_dt"))
         ), -1
    )
)
df.display()

## Check nulls

In [0]:
test_nulls = df.where(
    df.prd_id.isNull() |
    df.prd_key.isNull() |
    df.prd_nm.isNull() |
    df.prd_cost.isNull() |
    df.prd_line.isNull() 
)
test_nulls.display()

## Check production cost

Production costs shouldn't be null, so they will be replaced with 0

In [0]:
df = df.withColumn("prd_Cost", F.coalesce("prd_cost", F.lit(0)))
df.display()

## Modify production lines

Modify abbreviation to their full name with business context

In [0]:
df = df.withColumn("prd_line", 
    F.when(df.prd_line == "T", "Touring")
    .when(df.prd_line == "S", "Other sales")
    .when(df.prd_line == "M", "Mountain")
    .when(df.prd_line == "R", "Road")
    .otherwise('n/a')
)
df.display()

## Modify product key

Table prd_key contains 2 foreign keys:
* To table erp_px_cat_g1v2, it contains id with "AC_BR" expression, which can be the first 5 letters of product key
* To table crm_sls_details, it contains sls_prd_key with "BK-R93R-62" expression, which can be rest of the letters in product key (starting from 7th character)

Create cat_id (category id) for erp_px_cat_g1v2 table, first 5 characters and replace "-" with "_"

In [0]:
df = df.withColumn(
    "cat_id", 
    F.regexp_replace(F.substring(df.prd_key, 1, 5), "-", "_")
)
df.display()

Modify prd_key (product key) to accommodate sls.details table by leaving out the first 6 characters

In [0]:
df = df.withColumn(
    "prd_key", 
    F.substring(df.prd_key, 7, length(df.prd_key))
)
df.display()

In [0]:
df.limit(100).display()

# Write table silver.crm_prd_info

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("db_project.silver.crm_prd_info")